# Gerar o pacote (zip) do app 3 — Mobilidade Urbana — v1
Roda o comando de download do ExB (`zipApp`, só pt-br) e copia o zip para `Documents\Saidas_Notebooks\Mobilidade_v1.zip`.
Antes: o app 3 tem que estar **publicado** no construtor (Salvar → Publicar).
Saída: `Documents\Saidas_Notebooks\Gerar_Zip_Mobilidade_v1.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, subprocess, shutil, glob, time, zipfile
NB_NOME = "Gerar_Zip_Mobilidade_v1"
EXB = r"C:\ExB_1.19"
APP_ID = "3"
ZIP_NOME = "Mobilidade_v1.zip"
home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
log("▶ CÉLULA 1 — Configuração: iniciando...")
pub = os.path.join(EXB, "server", "public", "apps", APP_ID, "config.json")
with open(pub, encoding="utf-8") as f:
    c = json.load(f)
if c.get("__not_publish") or not c.get("pages"):
    raise RuntimeError("O app 3 ainda não está publicado. No construtor: Salvar → Publicar, e rode de novo.")
log("  App 3 publicado — páginas:", ", ".join(p.get("label", k) for k, p in c["pages"].items()))
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Gerar o zip com o comando do ExB =====
log("\n▶ CÉLULA 2 — Geração do zip: iniciando (pode levar 1–3 minutos)...")
inicio = time.time()
js = "require('./server/src/middlewares/dev/apps/app-download.js').zipApp('%s', '%s', '', {locales:['pt-br']});" % (APP_ID, ZIP_NOME)
r = subprocess.run(["node", "-e", js], cwd=EXB, capture_output=True, text=True, shell=True, timeout=900)
if r.stdout.strip(): log("  saída:", r.stdout.strip()[-800:])
if r.stderr.strip(): log("  avisos:", r.stderr.strip()[-800:])
# o comando grava o zip de forma assíncrona: espera ele aparecer e parar de crescer
achado, tam_ant = None, -1
for _ in range(180):
    cands = [p for p in glob.glob(os.path.join(EXB, "**", ZIP_NOME), recursive=True) if os.path.getmtime(p) >= inicio - 5]
    if cands:
        p = max(cands, key=os.path.getmtime); t = os.path.getsize(p)
        if t == tam_ant and t > 0:
            achado = p; break
        tam_ant = t
    time.sleep(2)
if not achado:
    raise RuntimeError("Não encontrei o zip gerado. Confira se o ExB está aberto e mande este .txt.")
destino = os.path.join(PASTA_SAIDA, ZIP_NOME)
shutil.copy2(achado, destino)
with zipfile.ZipFile(destino) as z:
    nomes = z.namelist()
log(f"  Zip gerado em: {achado}")
log(f"  Copiado para:  {destino}  ({os.path.getsize(destino)/1e6:.1f} MB, {len(nomes)} arquivos)")
log("  Exemplo de conteúdo:", ", ".join(nomes[:6]))
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Próximo passo =====
log("\n▶ CÉLULA 3 — Próximo passo: iniciando...")
log("  Avise no chat: vou pedir acesso à pasta Saidas_Notebooks, pegar o Mobilidade_v1.zip e publicar no GitHub Pages.")
log("  Link previsto: https://jrnegrao.github.io/mobilidade/?locale=pt-br")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 3 concluída com sucesso.")
